In [7]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [5]:
conn = sqlite3.connect("../db.sqlite3")
cursor = conn.cursor()
query = '''
SELECT
    t1.id,
    t1.quantity,
    t1.order_id,
    t2.datetime,
    t1.product_id,
    t3.price,
    t3.name
FROM
    restaurant_orderitem AS t1
JOIN
    restaurant_order AS t2 ON t1.order_id = t2.id
JOIN
    restaurant_product AS t3 ON t1.product_id = t3.id;
'''
combined_df = pd.read_sql_query(query, conn)
print("DataFrame успешно создан:")
print(combined_df)
print(f"Общее количество строк в DataFrame: {len(combined_df)}")
combined_df.to_csv('combined_data.csv', index=False)
print("\nДанные успешно объединены и сохранены в файл 'combined_data.csv'.")

conn.close()

DataFrame успешно создан:
          id  quantity  order_id             datetime  product_id  price  \
0          1         2         1  2019-08-03 20:25:00           6   0.80   
1          2         1         1  2019-08-03 20:25:00         240  12.95   
2          3         1         1  2019-08-03 20:25:00          19   2.95   
3          4         1         1  2019-08-03 20:25:00          36   3.95   
4          5         1         1  2019-08-03 20:25:00         111   8.95   
...      ...       ...       ...                  ...         ...    ...   
74813  74814         2      3255  2016-05-18 18:53:00         247  17.95   
74814  74815         2     12905  2016-05-14 19:28:00         247  17.95   
74815  74816         2      3272  2016-05-10 13:18:00         247  17.95   
74816  74817         1     11404  2016-04-07 19:04:00         248  17.95   
74817  74818         1     11449  2015-10-02 17:29:00         247  17.95   

                        name  
0              Plain Papadum  

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:

df = pd.read_csv('combined_data.csv')
product_sales = df.groupby('name')['quantity'].sum()
top_10_products = product_sales.sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 8))
plt.pie(
    top_10_products,
    labels=top_10_products.index,
    autopct='%1.1f%%',
    startangle=140
)
plt.title('Процент популярности (по количеству) для 10 лучших продуктов')
plt.axis('equal')  # Обеспечивает круглую форму диаграммы
plt.savefig('top10_products_pie_chart.png')

print("\nКруговая диаграмма сохранена в файл 'top10_products_pie_chart.png'.")

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df = pd.read_csv('combined_data.csv')
df['Item Price'] = df['price'] * df['quantity']
print(df)

product_sales = df.groupby('name')['Item Price'].sum()
top_10_products = product_sales.sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 8))
plt.pie(
    top_10_products,
    labels=top_10_products.index,
    autopct='%1.1f%%',
    startangle=140
)
plt.title('оп-10 товаров по выручке')
plt.axis('equal')

print("\nКруговая диаграмма сохранена в файл 'top10_products_pie_chart.png'.")

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df = pd.read_csv('combined_data.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df['Item Price'] = df['price'] * df['quantity']
df['Order Hour'] = df['datetime'].dt.hour
hourly_revenue = df.groupby('Order Hour')['Item Price'].sum()

plt.figure(figsize=(12, 7))
hourly_revenue.plot(kind='bar', color='skyblue')
plt.title('Общий доход ресторана по часам дня')
plt.xlabel('Час заказа (от 0 до 23)')
plt.ylabel('Общая сумма заказов')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('hourly_revenue_chart.png')

print("\nГрафик сохранен в файл 'hourly_revenue_chart.png'.")

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df = pd.read_csv('combined_data.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df['Item Price'] = df['price'] * df['quantity']
df['Day of Week'] = df['datetime'].dt.day_name()
print(df)
daily_revenue = df.groupby('Day of Week')['Item Price'].sum()

plt.figure(figsize=(12, 7))
daily_revenue.plot(kind='bar', color='skyblue')
plt.title('Общий доход ресторана по дням')
plt.xlabel('День заказа (от ПН до ВС)')
plt.ylabel('Общая сумма заказов')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('Daily_revenue_chart.png')

print("\nГрафик сохранен в файл 'Daily_revenue_chart.png'.")